# Task 01 — Bronze: Raw Orders Load

**Workshop: Final Pipeline | Layer 1 of 3**

## Goal

Load `orders_batch.json` from the Volume into a Bronze Delta table.
Bronze = raw data + metadata only. No business logic.

```
Volume: DATASET_PATH/orders/orders_batch.json
            |
            v  spark.read + explicit schema
            |  + _load_ts (timestamp)  + _source_file (path)
            v
    bronze.lab_orders
```

## Expected schema

| Column | Type | Notes |
|--------|------|-------|
| order_id | StringType | Not null |
| customer_id | StringType | |
| product_id | StringType | |
| store_id | StringType | |
| order_datetime | StringType | Kept as String — Silver casts it |
| quantity | IntegerType | |
| unit_price | DoubleType | |
| discount_percent | DoubleType | |
| total_amount | DoubleType | |
| payment_method | StringType | |
| status | StringType | |
| _load_ts | TimestampType | Added: load time |
| _source_file | StringType | Added: source file path |

> Bronze rule: do not transform, calculate, or filter — that is Silver's job.


In [ ]:
%run ../../setup/00_setup

In [ ]:
dbutils.widgets.text("source_path", DATASET_PATH, "Source Path")
dbutils.widgets.text("catalog",     CATALOG,       "Catalog")
dbutils.widgets.text("schema",      BRONZE_SCHEMA, "Bronze Schema")

source_path  = dbutils.widgets.get("source_path")
catalog      = dbutils.widgets.get("catalog")
schema       = dbutils.widgets.get("schema")

orders_path  = f"{source_path}/orders/orders_batch.json"
target_table = f"{catalog}.{schema}.lab_orders"

print(f"Source : {orders_path}")
print(f"Target : {target_table}")

---

## Exercise 1 — Imports

Import `StructType`, `StructField`, and column type classes for the 11 JSON columns.
Also import `current_timestamp` and `input_file_name` from `pyspark.sql.functions`.


In [ ]:
# HINT -- imports:
#
#   from pyspark.sql.types import (
#       StructType, StructField,
#       StringType, IntegerType, DoubleType
#   )
#   from pyspark.sql.functions import current_timestamp, input_file_name
#
#   current_timestamp()
#       -> Column expression: server timestamp at execution time.
#          Records when the batch was loaded.
#
#   input_file_name()
#       -> Column expression: full path of the file being read.
#          Only resolves during a spark.read — empty string otherwise.

# YOUR CODE HERE
raise NotImplementedError("Complete this cell before proceeding")

---

## Exercise 2 — Define the schema

Define a `StructType` for the **11 JSON columns** listed in the table above.
`order_id` must have `nullable=False`; all others `nullable=True`.


In [ ]:
# HINT -- StructType:
#
#   schema = StructType([
#       StructField("column_name", DataType(), nullable=True),
#       ...
#   ])
#
#   Available types for this dataset:
#       StringType()    IntegerType()    DoubleType()
#
#   Keep order_datetime as StringType.
#   Casting to TimestampType is Silver's responsibility.

# YOUR CODE HERE
raise NotImplementedError("Complete this cell before proceeding")

orders_schema  # display

---

## Exercise 3 — Read the JSON file and add metadata columns

Read `orders_path` using your schema.
Add `_load_ts` and `_source_file` with `.withColumn()`.
Name the result `orders_raw`.


In [ ]:
# HINT -- spark.read for JSON:
#
#   df = (
#       spark.read
#           .format("json")       # JSON Lines: one object per line
#           .schema(schema_var)   # pass your StructType
#           .load("path")
#           .withColumn("_load_ts",     current_timestamp())
#           .withColumn("_source_file", input_file_name())
#   )
#
#   Always pass explicit schema -- inferSchema costs a full extra scan.

# YOUR CODE HERE
raise NotImplementedError("Complete this cell before proceeding")

In [ ]:
print(f"Rows loaded : {orders_raw.count():,}")
orders_raw.printSchema()
display(orders_raw.limit(5))

---

## Exercise 4 — Write to Bronze as a managed Delta table


In [ ]:
# HINT -- DataFrame.write:
#
#   df.write
#       .format("delta")
#       .mode("overwrite")                   # full refresh each run
#       .option("overwriteSchema", "true")   # allow schema changes on re-run
#       .saveAsTable("catalog.schema.table") # managed UC table
#
#   mode options:
#       "overwrite"     -> replace all data
#       "append"        -> add rows (incremental)
#       "errorIfExists" -> fail if table already exists

# YOUR CODE HERE
raise NotImplementedError("Complete this cell before proceeding")

In [ ]:
row_count = spark.table(target_table).count()
print(f"OK  {target_table}  ->  {row_count:,} rows")
spark.sql(f"DESCRIBE HISTORY {target_table}")\
    .select("version","timestamp","operation")\
    .show(3, truncate=False)

In [ ]:
import json
dbutils.notebook.exit(json.dumps({"status":"SUCCESS","table":target_table,"rows":row_count}))